# Demo: ledger2 eval

When does a result have to be recomputed? `rev.eval(name, **keys)`
runs a module function against a revision, records everything the
function read, and seals the result as a memo entry. The next call
with the same keys is a hit while everything it read still reads
the same, and a miss once any of it moves. This notebook trains the
identity model for games G5 and G1 with `svp.identity.train` and
watches what turns a hit into a miss.

In [1]:
import inspect, os, shutil, sys, tempfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, "demo_svp")          # the stand-in svp package
from dvc_dat.ledger2 import Code, Env, Ledger, NotEvaluated
from dvc_dat.ledger2 import bump, set_env_provider
from svp.shots import Shots
import svp.identity

ENV = Env({"sv-pipe": Code("0.2.2", "7d742989")},
          lock="sha256:5e1f", platform="darwin-arm64-py3.11")
old_provider = set_env_provider(lambda module: ENV)
ledger_dir = Path(tempfile.mkdtemp(prefix="demo-l2-eval-"))
os.environ["DAT_LEDGER"] = str(ledger_dir)
L = Ledger.open()
with L.transaction(by="human:dan") as t:
    t.add("code/sv-pipe", "version", "0.2.2")
    t.add("code/sv-pipe", "commit", "7d742989")
    t.add("model/player-id", "version", "2.1.0")
    t.add("data/norm", "version", "7.0.0")
    for g, n in (("G5", 5), ("G1", 3)):
        frames = pd.DataFrame({"frame": range(n)})
        t.add(f"run/0-2-2/{g}", "value", Shots(frames))
print(inspect.getsource(svp.identity.train))

def train(rev, *, game: str, epochs: int = 8):
    base = rev["model/player-id", "version"]
    run = rev.load(f"run/0-2-2/{game}")
    acc = 0.93 if game == "G5" else 0.88
    return {"model": f"identity-{game}-{epochs}", "base": base,
            "frames": len(run.table), "acc": acc}



The function takes the revision first and its keys after. It reads
a version row and loads a run, both through `rev`. The environment
is pinned to sv-pipe 0.2.2 at `7d742989`, matching the ledger's
`code/sv-pipe` rows, which is what lets it run.

## A miss runs and seals; a hit writes nothing

In [2]:
rev = L.at()
m = rev.eval("svp.identity.train", game="G5")
print(m.obj)
print(m.subject, m.sealed, m.rev, "top:", L.at())

{'model': 'identity-G5-8', 'base': '2.1.0', 'frames': 5, 'acc': 0.93}
Pickled/1ffae442f3e9444b True Rev(1) top: Rev(2)


The first call missed: it ran, sealed the result as `Pickled/...`
in a new revision (2), and handed back the value itself with its
provenance on it. `m.rev` is the revision it was evaluated at.

In [3]:
hit = L.at().eval("svp.identity.train", game="G5")
print(hit.subject == m.subject, "top:", L.at())
for item, value in m.trace.items():
    print(f"  {item:30} {value[:26]}")

True top: Rev(2)
  model/player-id                2.1.0
  dat:run/0-2-2/G5               sha256:890d9810f298b6bca59
  code/sv-pipe                   0.2.2+7d742989
  fn                             ledger/fn/svp.identity.tra
  uv.lock                        sha256:5e1f
  platform                       darwin-arm64-py3.11


The second call hit: same subject, no new revision. The trace is
what the hit test compares: the two reads (`model/player-id`, the
run's hash), the code, the function's own subject, the lock and the
platform. `data/norm` is not in it: the function never read it.

## `memo="get"` and `memo=False`

In [4]:
rev = L.at()
print(rev.eval("svp.identity.train", game="G1", memo="get"))
c = rev.eval("svp.identity.train", game="G1", memo=False)
print(c.obj, c.sealed, c.subject, "top:", L.at())

None
{'model': 'identity-G1-8', 'base': '2.1.0', 'frames': 3, 'acc': 0.88} False None top: Rev(2)


`"get"` only looks: `None` for G1, never computed. `False` runs and
seals nothing.

## `load_fn`: resolve once, call many

In [5]:
f = L.at().load_fn("svp.identity.train")
print(f.name, f.mode, f.signature)
print(f.subject)
for game in ("G5", "G1"):
    print(game, f(game=game).subject)
print(f(game="G5", epochs=8).subject == m.subject)

svp.identity.train base (*, game: str, epochs: int = 8)
ledger/fn/svp.identity.train/088690e91a162df2
G5 Pickled/1ffae442f3e9444b
G1 Pickled/27c6c360d31862a4
True


`load_fn` returns the `Fn`; calling it is `eval` at the revision it
was loaded at. G5 hits and G1 runs. A key passed at its default is
the same entry.

## What turns a hit into a miss

In [6]:
with L.transaction(by="human:dan") as t:
    bump(t, "data/norm", "patch")
r = L.at().eval("svp.identity.train", game="G5")
print("after a data/norm bump: ", r.subject == m.subject)

with L.transaction(by="human:dan") as t:
    bump(t, "model/player-id", "patch")
r2 = L.at().eval("svp.identity.train", game="G5")
print("after a player-id bump:", r2.subject == m.subject)
print(r2.obj["base"], m.diff(r2))

after a data/norm bump:  True
after a player-id bump: False
2.1.1 {'model/player-id': ('2.1.0', '2.1.1')}


A bump of something the function never read leaves it a hit; a bump
of something it read is a miss, a run and a new entry. `diff` names
exactly the item that moved.

In [7]:
with L.transaction(by="human:dan") as t:
    bump(t, "code/sv-pipe", "patch")
    t.add("code/sv-pipe", "commit", "b48dca7c")
try:
    L.at().eval("svp.identity.train", game="G5")
except NotEvaluated as e:
    print("NotEvaluated:", e)

NotEvaluated: svp.identity.train: rev 7 binds code/sv-pipe to 0.2.3+b48dca7c; this environment is 0.2.2+7d742989


A code bump in the ledger is a miss, and this environment is still
0.2.2 at `7d742989`: rather than seal a result with false
provenance, `eval` refuses to run.

In [8]:
NEW = Env({"sv-pipe": Code("0.2.3", "b48dca7c")},
          lock=ENV.lock, platform=ENV.platform)
set_env_provider(lambda module: NEW)
r3 = L.at().eval("svp.identity.train", game="G5")
print(r2.diff(r3))

{'code/sv-pipe': ('0.2.2+7d742989', '0.2.3+b48dca7c')}


In an environment at the new commit it runs, and the trace says the
code moved.

## Stale entries are kept

In [9]:
top = L.at()
for e in top.refs(fn="svp.identity.train", game="G5"):
    rows = dict(e.rev.all(e.subject))
    print(e.subject, rows["trace.model/player-id"],
          rows["trace.code/sv-pipe"])
first = L.at(1).eval("svp.identity.train", game="G5", memo="get")
print("rev 1 still answers:", first.subject == m.subject)

Pickled/1ffae442f3e9444b 2.1.0 0.2.2+7d742989
Pickled/5faac03c0d315185 2.1.1 0.2.2+7d742989
Pickled/ee371bbb23078f02 2.1.1 0.2.3+b48dca7c
rev 1 still answers: True


Every entry stays, one per trace. A revision that still holds the
old bindings, here revision 1, still finds its own entry.

In [10]:
set_env_provider(old_provider)
shutil.rmtree(ledger_dir)
del os.environ["DAT_LEDGER"]
print("removed:", not ledger_dir.exists())

removed: True
